# Project 32 — LangGraph Multi-Step Sales Research Flow
## Company Lookup → Analysis → Outreach Draft

**Stack:** LangGraph · Ollama · Jupyter

In [ ]:
# !pip install -q langgraph langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.3)

class SalesState(TypedDict):
    company_name: str
    company_profile: str
    pain_points: str
    value_proposition: str
    outreach_email: str

## Step 2 — Define Sales Research Nodes

In [ ]:
def research_company(state: SalesState) -> SalesState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a sales researcher. Create a company profile including: "
         "industry, estimated size, likely tech stack, and recent news/trends. "
         "Use your knowledge to make reasonable inferences."),
        ("human", "Research this company: {company}")
    ])
    chain = prompt | llm | StrOutputParser()
    profile = chain.invoke({"company": state["company_name"]})
    print(f"  📊 Company profile researched")
    return {"company_profile": profile}

def identify_pain_points(state: SalesState) -> SalesState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Based on the company profile, identify 3-5 likely pain points "
         "that our AI/ML platform could solve. Be specific to their industry."),
        ("human", "Company: {company}\nProfile: {profile}")
    ])
    chain = prompt | llm | StrOutputParser()
    pains = chain.invoke({"company": state["company_name"], "profile": state["company_profile"]})
    print(f"  🎯 Pain points identified")
    return {"pain_points": pains}

def craft_value_prop(state: SalesState) -> SalesState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Create a tailored value proposition that addresses the identified "
         "pain points. Focus on measurable outcomes (ROI, time saved, etc.)"),
        ("human", "Company: {company}\nPain Points: {pains}")
    ])
    chain = prompt | llm | StrOutputParser()
    vp = chain.invoke({"company": state["company_name"], "pains": state["pain_points"]})
    print(f"  💎 Value proposition crafted")
    return {"value_proposition": vp}

def draft_outreach(state: SalesState) -> SalesState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", """Write a personalized cold outreach email. Rules:
- Subject line that creates curiosity
- Opening that references something specific about their company
- 2-3 sentences connecting their pain to our solution
- Clear CTA (15-min call)
- Under 150 words
- Professional but not corporate"""),
        ("human", """Company: {company}
Profile: {profile}
Pain Points: {pains}
Value Prop: {vp}

Write the outreach email.""")
    ])
    chain = prompt | llm | StrOutputParser()
    email = chain.invoke({
        "company": state["company_name"], "profile": state["company_profile"],
        "pains": state["pain_points"], "vp": state["value_proposition"],
    })
    print(f"  ✉️ Outreach email drafted")
    return {"outreach_email": email}

## Step 3 — Build Sales Pipeline Graph

In [ ]:
graph = StateGraph(SalesState)
graph.add_node("research", research_company)
graph.add_node("pain_points", identify_pain_points)
graph.add_node("value_prop", craft_value_prop)
graph.add_node("outreach", draft_outreach)

graph.set_entry_point("research")
graph.add_edge("research", "pain_points")
graph.add_edge("pain_points", "value_prop")
graph.add_edge("value_prop", "outreach")
graph.add_edge("outreach", END)

app = graph.compile()
print("Sales research pipeline compiled!")

## Step 4 — Run for Target Companies

In [ ]:
companies = ["Stripe", "Shopify", "Datadog"]

for company in companies:
    print(f"\n{'='*60}")
    print(f"RESEARCHING: {company}")
    print("="*60)
    result = app.invoke({
        "company_name": company, "company_profile": "", "pain_points": "",
        "value_proposition": "", "outreach_email": "",
    })
    print(f"\n--- OUTREACH EMAIL ---")
    print(result["outreach_email"])

## What You Learned
- **Multi-step sales research** pipeline with LangGraph
- **Progressive context building** across nodes
- **Personalized outreach** from automated research